# A3 Stage 2: open-weight models on a GPU

Runs the open models (`Qwen2.5-VL-7B-Instruct`, `InternVL3-8B`, `MiniCPM-V-4_5`) on the four
benchmarks with exactly the same code, item lists and settings as the API models
(`experiment/run.py`). Output: `a3_output.zip` with `raw/rerun/...` and `logs/` — merge it
into the repository with `python experiment/merge_logs.py <unzipped dir>`.

**GPU:** VLMEvalKit loads these models in bfloat16 and Qwen2.5-VL with FlashAttention-2,
so the GPU must support both (Ampere or newer: RTX 3090/4090, A10, L4, A100, H100).
Kaggle's free T4/P100 do not, and the run is not changed to make them fit.
Works on any Jupyter with CUDA (RunPod, Vast.ai, Colab, Kaggle with a suitable GPU).
Internet must be on. Nothing in this notebook needs an API key.


In [ ]:
# ---- parameters ------------------------------------------------------------
REPO_URL = "https://github.com/<your-account>/<a3-repo>.git"   # the public A3 repository
MODEL    = "Qwen2.5-VL-7B-Instruct"     # or InternVL3-8B, MiniCPM-V-4_5
SPLIT    = "pilot"                       # pilot first, then main
BENCHES  = ["OCRBench_v2", "CCOCR", "OmniDocBench", "olmOCRBench"]
OPERATOR = "Miras"                       # or Adilzhan
VLMEVALKIT_COMMIT = "f57c3d90154010ba176fbafc39bb79ee521fbf70"

In [ ]:
import os, subprocess, sys
W = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
os.chdir(W)
def sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)
if not os.path.isdir("a3"):
    sh(f"git clone {REPO_URL} a3")
if not os.path.isdir("a3/third_party/VLMEvalKit"):
    sh("git clone https://github.com/open-compass/VLMEvalKit.git a3/third_party/VLMEvalKit")
sh(f"cd a3/third_party/VLMEvalKit && git checkout -q {VLMEVALKIT_COMMIT} && git rev-parse HEAD")
sh("pip install -q -e a3/third_party/VLMEvalKit -r a3/experiment/requirements-extra.txt")
sh("python -m playwright install --with-deps chromium")   # olmOCR-Bench math tests
sh("pip install -q flash-attn --no-build-isolation")       # Qwen2.5-VL uses attn_implementation=flash_attention_2

In [ ]:
# GPU and library versions go into raw/.../run_meta.json automatically; print them here too.
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
import torch, transformers
print("python", sys.version.split()[0], "torch", torch.__version__, "cuda", torch.version.cuda,
      "transformers", transformers.__version__)
print("bf16 supported:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

In [ ]:
# Benchmark TSVs (about 6 GB) with md5 check against the values pinned in VLMEvalKit.
os.environ["LMUData"] = f"{W}/LMUData"
sh("cd a3 && python experiment/download_data.py")

In [ ]:
# The runs. Each benchmark appends one row to a3/logs/run_log.csv (also on failure).
os.environ["HF_HOME"] = f"{W}/hf"
for bench in BENCHES:
    sh(f"cd a3 && python experiment/run.py --model {MODEL} --bench {bench} --split {SPLIT} --operator {OPERATOR}")

In [ ]:
sh("cd a3 && tail -n 8 logs/run_log.csv")
sh(f"cd a3 && zip -qr {W}/a3_output.zip logs raw/rerun -x '*.pkl' '*_subset_gt.tsv'")
print("download", f"{W}/a3_output.zip")